# 06 · Reconstruct results and estimate uncertainty


In [ ]:
from pathlib import Path
import json, os, sys

# Find the checkout/release from the notebook's working directory.
ROOT = Path(os.environ.get('GF_ROOT', Path.cwd())).resolve()
while not (ROOT / 'src/gavd6_sjepa').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'src/gavd6_sjepa').is_dir(), 'Open this notebook from the GAVD6 checkout or release.'
sys.path.insert(0, str(ROOT / 'notebooks/gait_fidelity'))
sys.path.insert(0, str(ROOT / 'src'))
from tutorial_helpers import configure, preview_images
study = configure(ROOT)


## 1 · Verify the retained evidence before summarizing it

The canonical verifier checks the frozen configuration and source, retained
artifact hashes, and reconstruction of each trained model's saved `A_error`,
`waveform_error`, and `support_frames`. Notebook 05 separately reconstructs
position/displacement and the three contrasts for a metadata-selected example.
Below we expose the remaining calculation from records to people and seeds.

A byte-for-byte check establishes that files have not changed since their
receipt. It cannot establish that the study's assumptions are scientifically
correct, so numerical reconstruction and methodological checks remain separate.


In [ ]:
study.command('verify')
study.command('report')


In [ ]:
import hashlib
import numpy as np
import pandas as pd
from IPython.display import Markdown, display, Image
from io import BytesIO
import matplotlib.pyplot as plt

config = study.artifact('config.json')
plan = study.artifact('plan.json')
ledger = study.artifact('ledger.json')
frozen = study.artifact('frozen.json')

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

assert file_sha256(study.work / 'config.json') == frozen['config_sha256']
assert file_sha256(study.work / 'plan.json') == frozen['plan_sha256']
phase = next(p for p in plan['phases'] if p['phase'] != 'pretrain')
completed = ledger['completed'][phase['phase_id']]
assert file_sha256(completed['receipt']) == completed['sha256']
receipt = json.loads(Path(completed['receipt']).read_text())
artifact_checks = []
for path, expected in receipt['artifacts'].items():
    actual = file_sha256(path)
    assert actual == expected, f'Changed artifact: {path}'
    artifact_checks.append(dict(artifact=Path(path).name, sha256=actual[:16], matches=True))
display(pd.DataFrame(artifact_checks))
print('Illustrated phase:', phase['phase_id'], '| canonical verifier checks all completed phases.')


## 2 · Average conditions, windows, raw motions and people

An extractor, view, movement variant or repeat seed does not create a new
participant. Let \(e_{psfc}\) be an error for person \(p\), seed \(s\), source
family \(f\), and condition \(c\). The saved summary first computes each
family's mean over its supported conditions, averages windows belonging to the
same raw motion, then gives each supported motion equal weight within a person:

\[
\bar e_{psf}=\operatorname{mean}_{c}(e_{psfc}),\qquad
\bar e_{psm}=\operatorname{mean}_{f\in m}(\bar e_{psf}),\qquad
\bar e_{ps}=\operatorname{mean}_{m}(\bar e_{psm}).
\]

The code follows pandas' missing-value convention: reference-ineligible rows
are skipped by these means, and an entirely unsupported group remains missing.
Prediction failures stay in the mean through their declared penalties. A result
therefore describes the **reference-supported population** and must be reported
with coverage; it is not an unconditional claim about every prepared recording.


In [ ]:
saved_person = pd.read_csv(study.artifact('evaluation/per-person.csv'))
from gavd6_sjepa.research_directions.gait_fidelity.data import load_dataset
bundle = load_dataset(study.bundle_path())
motion_for_family = {r['source_family_id']: r['motion_hash'] for r in bundle.records}
keys = ['method', 'seed', 'canonical_person_id']
metrics = ['A_error', 'waveform_error', 'visible_nle', 'synthetic_all_nle',
           'displacement_nle', 'assignment_failure_rate']

def reconstruct_people(relative, metrics, exclude_no_change=False):
    # Sums and nonmissing counts permit bounded-memory CSV reads. Averaging
    # chunk means would overweight a small fragment at a chunk boundary.
    sums, counts = [], []
    family_keys = keys + ['source_family_id']
    for frame in pd.read_csv(study.artifact(relative), chunksize=10000):
        if exclude_no_change: frame = frame.loc[frame.movement_state.ne('no_change')]
        grouped = frame.groupby(family_keys)[metrics]
        sums.append(grouped.sum())
        counts.append(grouped.count())
    total = pd.concat(sums).groupby(level=family_keys).sum()
    supported = pd.concat(counts).groupby(level=family_keys).sum()
    families = (total / supported.replace(0, np.nan)).reset_index()
    families['motion_hash'] = families.source_family_id.map(motion_for_family)
    assert families.motion_hash.notna().all()
    motions = families.groupby(keys + ['motion_hash'])[metrics].mean()
    return motions.groupby(keys).mean().reset_index()

person_means = reconstruct_people('evaluation/per-window.csv', metrics)
for relative, metric, exclude in [('responses', 'response_error', True),
                                 ('nuisance', 'nuisance_error', False),
                                 ('interaction', 'interaction_error', True)]:
    values = reconstruct_people('evaluation/' + relative + '.csv', [metric], exclude)
    person_means = person_means.merge(values, on=keys, how='left', validate='one_to_one')

reconstructed = person_means.set_index(keys).sort_index()
retained = saved_person.set_index(keys).sort_index()
assert reconstructed.index.equals(retained.index)
np.testing.assert_allclose(reconstructed.to_numpy(float), retained[reconstructed.columns].to_numpy(float),
                           atol=1e-8, rtol=1e-7, equal_nan=True)
print('All person-level metrics reconstruct:', len(reconstructed), 'person × seed × method rows')
display(person_means.head(12))


## 3 · Define the primary paired comparison before reading its outcome

The saved configuration specifies the candidate and comparator. For a
lower-is-better error, define the person-and-seed improvement as

\[
D_{ps}=\bar e_{ps}^{\mathrm{comparator}}-\bar e_{ps}^{\mathrm{candidate}}.
\]

Positive values favor the candidate. Subtraction occurs **within the same
person and seed**, rather than between two unrelated collections of scores.
The point estimate is the mean of the complete person-by-seed matrix. A missing
method, unsupported value, or incomplete seed grid blocks the uncertainty
calculation rather than silently reducing the comparison population.


In [ ]:
evaluation = config['evaluation']
candidate = evaluation['primary_candidate']
comparator = evaluation['primary_comparator']
metric = 'response_error'
selected = person_means.loc[person_means.method.isin([candidate, comparator])]
wide = selected.pivot(index=['canonical_person_id', 'seed'], columns='method', values=metric)
has_methods = candidate in wide and comparator in wide
complete_pairs = has_methods and not wide.isna().any().any()
delta = (wide[comparator] - wide[candidate]).unstack('seed') if complete_pairs else pd.DataFrame()
supported = complete_pairs and len(delta) >= 2 and not delta.isna().any().any()
print('Candidate:', candidate, '\nComparator:', comparator)
print('Complete person/seed grid:', supported)
display(delta)


## 4 · Reproduce the crossed person-and-seed bootstrap

A **bootstrap** approximates sampling variation by repeatedly sampling observed
units with replacement. We draw whole people, keeping their correlated families
and conditions together. We also draw fitted seeds as a second, crossed factor:
the same sampled seed columns apply to every sampled person. Methods stay paired
because we resample their precomputed differences.

For each draw, the person-conditional estimate averages the sampled people over
the original seed columns. The crossed estimate additionally resamples those
columns. The 2.5th and 97.5th percentiles of these means form the two displayed
95% intervals. We reproduce the production random-number draw order exactly;
the demonstration does not consume the training random-number stream.

With only three fitted seeds, the empirical seed distribution remains sparse.
These intervals describe variation in the available development data and fits;
they do not account for earlier method selection, dataset screening decisions,
or reuse of development people. Crossing zero is not the sole criterion for a
scientifically useful or generalizable result.


In [ ]:
saved_comparison = study.artifact('evaluation/comparisons.json')[metric]
if supported:
    values = delta.to_numpy(float)  # [people, fitted seeds]
    rng = np.random.default_rng(evaluation['bootstrap_seed'])
    person_draws, crossed_draws = [], []
    for _ in range(evaluation['bootstrap_draws']):
        sampled_people = rng.integers(len(values), size=len(values))
        sampled_seeds = rng.integers(values.shape[1], size=values.shape[1])
        person_draws.append(float(values[sampled_people].mean()))
        crossed_draws.append(float(values[sampled_people][:, sampled_seeds].mean()))
    improvement = float(values.mean())
    person_interval = np.quantile(person_draws, [0.025, 0.975])
    crossed_interval = np.quantile(crossed_draws, [0.025, 0.975])
    np.testing.assert_allclose(improvement, saved_comparison['improvement'], atol=1e-8)
    np.testing.assert_allclose(person_interval, saved_comparison['person_conditional_ci95'], atol=1e-8)
    np.testing.assert_allclose(crossed_interval, saved_comparison['crossed_person_seed_ci95'], atol=1e-8)
    display(pd.DataFrame([
        dict(summary='Person-conditional', improvement=improvement, lower=person_interval[0], upper=person_interval[1]),
        dict(summary='Crossed person and seed', improvement=improvement, lower=crossed_interval[0], upper=crossed_interval[1]),
    ]))
    print('Per-seed improvements:', delta.mean(axis=0).to_dict())
else:
    assert saved_comparison['status'].startswith('insufficient')
    print('The fixed comparison is unsupported:', saved_comparison)


### Inspect whether the average hides inconsistent people or seeds

Each dot below represents the same person's paired improvement for one fitted
seed, averaged over that person's supported source families. The line at zero
separates improvement from harm. These dots are related observations; their
number is not the evaluation sample size. Deterministic baselines have identical
predictions repeated across seed rows for bookkeeping, not independently fitted
calibrations.


In [ ]:
if supported:
    fig, ax = plt.subplots(figsize=(9, max(3.5, .36 * len(delta) + 1.5)), layout='constrained')
    vertical = np.arange(len(delta))
    offsets = np.linspace(-.18, .18, delta.shape[1])
    for j, (seed_value, offset) in enumerate(zip(delta.columns, offsets)):
        ax.scatter(delta.iloc[:, j], vertical + offset, s=28, label=f'Seed {seed_value}')
    ax.axvline(0, color='#777777', linewidth=1)
    ax.set_yticks(vertical, delta.index.astype(str))
    ax.set(xlabel='Comparator error − candidate error (degrees; positive favors candidate)',
           ylabel='Development person',
           title=('Software fixture: ' if study.fixture else 'Development study: ')
                 + 'movement-response improvement for each person and seed')
    ax.legend(frameon=False, ncol=min(3, delta.shape[1]))
    buffer = BytesIO(); fig.savefig(buffer, format='png', dpi=130); plt.close(fig)
    display(Image(data=buffer.getvalue()))


## 5 · Read the report with a claim-specific evidence checklist

The final report is generated from the retained summaries, not from these
teaching calculations. The equality checks above show that the tutorial follows
the same rules. A reproducible result can still be limited by small or previously
inspected populations and by the difference between synthetic and real references.

| Intended conclusion | Required comparison and evidence |
| --- | --- |
| Restoration improves position accuracy | Same-population coordinate error and coverage against unchanged poses and training-only calibration |
| Change supervision preserves movement | Matched base versus paired-change objectives, with identical endpoint exposure and useful effect sizes |
| Anatomical masking contributes | Graph masks versus topology/duration controls with acceptable coverage and run-length matching |
| JEPA contributes useful features | Matched coordinate pretraining, initialized encoder and shuffled-reference controls, plus direct training |
| Meaningful pairing contributes | Correct pairing versus per-example labels and valid re-pairing, with the distribution audit reported |
| The method generalizes | A frozen protocol evaluated on new people or conditions, with reference coverage and prediction failures retained |
| The method supports clinical gait assessment | Independent clinical measurements and an appropriate real-world population |

This report uses the saved development population; some people may have been
inspected in earlier studies. The original manifest test split remains locked
until the protocol and exposure review are fixed. A complete fixture or a
favorable interval does not establish independent confirmation. The data are image-plane
proxies for gait measurements; a result cannot establish preservation of disease
severity, anatomical 3D range of motion or clinical change without additional
reference evidence. Additional entries in the selected core or full matrix
provide attribution and exploratory comparisons under the same primary claim.


In [ ]:
summary = study.artifact('evaluation/summary.json')
print(json.dumps({k: summary[k] for k in ('status', 'evidence_status', 'scientific_gate',
                 'independent_confirmation', 'clinical_validation')}, indent=2))
display(Markdown(study.artifact('report.md').read_text()))


## 6 · Retain enough evidence for another reader to reconstruct the result

On your Mac, use `slurm/gait-fidelity/retrieve.sh` with the HAIC work directory
and a new local destination. That helper copies predictions, tables and viewer
artifacts while leaving model checkpoints on HAIC. Full artifact-hash verification
also needs the checkpoints; the Slurm guide includes the complete-transfer
command. Preserve the original release and configuration rather than replacing
them with a newer implementation under the same run name.

Write the evaluated participant count, retained source-family count, fitted
seeds, reference coverage, prediction failure rate, effect size and uncertainty
alongside the comparison. State which decisions used development outcomes and
which conditions were genuinely held out. Report unresolved masking or pairing
audit failures as limits on the corresponding interpretation.
